# Compare Ecowitt data with reference sensors

This notebook is designed for **Google Colab**.

It does four things:

1. Clones or reads your **gothic** repository.
2. Loads **Ecowitt** data from the repo.
3. Loads the **reference sensor** file from the repo.
4. Aligns timestamps, compares selected variables, computes metrics, and creates plots.

Update the configuration cell first, then run the notebook from top to bottom.

In [1]:
#@title 1) Install / import packages
import os
import subprocess
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

## Configuration

Edit the values below before running.

Notes:

- `REPO_URL` should point to your repository.
- `ECOWITT_FILE` and `REFERENCE_FILE` must be paths **inside the repo**.
- `VARIABLE_MAP` links the Ecowitt column name to the matching reference column name.
- If your Ecowitt file has no header, the notebook can assign default fallback names.

In [33]:
#@title 2) User configuration

# --- Repository ---
REPO_URL = "https://github.com/paulmunozpauta/VLIR_SI_EC_2025-2027"   # change this
REPO_BRANCH = "main"                                   # change if needed
REPO_DIR = "/content/VLIR_SI_EC_2025-2027"              # change this

# Set to False if the repo is already available in /content
CLONE_REPO = True

# --- Files inside the repo ---
ECOWITT_FILE = "datasets/AW005.csv"                # change this
REFERENCE_FILE = "data/reference/reference_sensor.csv" # change this

# Time columns
ECOWITT_TIME_COL = "timestamp"
REFERENCE_TIME_COL = "Timestamp"   # change to the real reference time column

# Set timezone if needed, for example "Europe/Brussels"
TIMEZONE = None

# Common comparison timestep
RESAMPLE_FREQ = "1H"   # examples: "5min", "15min", "1H"

# Variable mapping
# Change only the "reference" names to your real column names
VARIABLE_MAP = {
    "temp_c": {
        "ecowitt": "outdoor_temp",
        "reference": "temp_ref"
    },
    "humidity_pct": {
        "ecowitt": "outdoor_humidity",
        "reference": "rh_ref"
    },
    "dew_point_c": {
        "ecowitt": "dew_point",
        "reference": "dew_point_ref"
    },
    "feels_like_c": {
        "ecowitt": "feels_like",
        "reference": "feels_like_ref"
    },
    "wind_speed": {
        "ecowitt": "wind_speed",
        "reference": "wind_speed_ref"
    },
    "wind_gust": {
        "ecowitt": "wind_gust",
        "reference": "wind_gust_ref"
    },
    "pressure": {
        "ecowitt": "pressure",
        "reference": "pressure_ref"
    },
    "rain_rate": {
        "ecowitt": "rain_rate",
        "reference": "rain_rate_ref"
    },
    "daily_rain": {
        "ecowitt": "daily_rain",
        "reference": "daily_rain_ref"
    },
    "solar_radiation": {
        "ecowitt": "solar_radiation",
        "reference": "solar_ref"
    },
    "uv_index": {
        "ecowitt": "uv_index",
        "reference": "uv_ref"
    },
}


In [3]:
#@title 3) Clone repo (skip if not needed)

if CLONE_REPO:
    if os.path.exists(REPO_DIR):
        print(f"Repository folder already exists: {REPO_DIR}")
    else:
        cmd = ["git", "clone", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR]
        print("Running:", " ".join(cmd))
        subprocess.run(cmd, check=True)
else:
    print(f"Using existing repo at: {REPO_DIR}")

print("Repo dir:", REPO_DIR)

Running: git clone --branch main https://github.com/paulmunozpauta/VLIR_SI_EC_2025-2027 /content/VLIR_SI_EC_2025-2027
Repo dir: /content/VLIR_SI_EC_2025-2027


In [51]:
#@title 4) Helper functions

DEFAULT_ECOWITT_COLUMNS_15 = [
    "timestamp",
    "outdoor_temp",
    "outdoor_humidity",
    "dew_point",
    "feels_like",
    "wind_speed",
    "wind_gust",
    "wind_dir",
    "pressure",
    "rain_rate",
    "daily_rain",
    "solar_radiation",
    "uv_index",
    "indoor_temp",
    "indoor_humidity",
]


def robust_read_csv(path):
    """
    Reads a CSV and tries to detect whether it has a header.
    If the first field looks like a timestamp and there are exactly 15 columns,
    fallback Ecowitt names are assigned.
    """
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")

    # First try normal header parsing
    df = pd.read_csv(path)

    # If the columns themselves look like data, reload without header
    first_col_name = str(df.columns[0])
    first_col_looks_like_time = pd.to_datetime(pd.Series([first_col_name]), errors="coerce").notna().iloc[0]

    if first_col_looks_like_time:
        df = pd.read_csv(path, header=None)
        if df.shape[1] == 15:
            df.columns = DEFAULT_ECOWITT_COLUMNS_15
        else:
            df.columns = [f"col_{i}" for i in range(df.shape[1])]

    # Clean BOM on first column if present
    df.columns = [str(c).replace("\\ufeff", "").strip() for c in df.columns]
    return df


def standardize_time(df, time_col, timezone=None):
    df = df.copy()
    if time_col not in df.columns:
        raise KeyError(f"Time column '{time_col}' not found. Available columns: {list(df.columns)}")

    df[time_col] = pd.to_datetime(df[time_col], errors="coerce", utc=True)
    df = df.dropna(subset=[time_col])

    if timezone is not None:
        df[time_col] = df[time_col].dt.tz_convert(timezone)

    df = df.sort_values(time_col).drop_duplicates(subset=[time_col])
    return df


def apply_date_filter(df, time_col, start_date=None, end_date=None):
    df = df.copy()
    if start_date is not None:
        df = df[df[time_col] >= pd.to_datetime(start_date, utc=True)]
    if end_date is not None:
        df = df[df[time_col] <= pd.to_datetime(end_date, utc=True)]
    return df


def convert_numeric_columns(df, exclude=None):
    df = df.copy()
    exclude = set(exclude or [])
    for col in df.columns:
        if col not in exclude:
            df[col] = pd.to_numeric(df[col], errors="ignore")
    return df


def resample_dataset(df, time_col, column_map, freq, agg_rules, dataset_name):
    """
    column_map is a dict like:
    {"temp": "outdoor_temp", "humidity": "outdoor_humidity"}
    """
    keep_cols = [time_col] + [c for c in column_map.values() if c in df.columns]
    tmp = df[keep_cols].copy()
    tmp = tmp.set_index(time_col)

    agg = {}
    rename_back = {}
    for logical_name, col in column_map.items():
        if col not in tmp.columns:
            print(f"[{dataset_name}] Missing column for '{logical_name}': {col}")
            continue
        agg[col] = agg_rules.get(logical_name, "mean")
        rename_back[col] = logical_name

    out = tmp.resample(freq).agg(agg)
    out = out.rename(columns=rename_back)
    out.index.name = time_col
    return out.reset_index()


def compute_metrics(y_true, y_pred):
    mask = y_true.notna() & y_pred.notna()
    y_true = y_true[mask]
    y_pred = y_pred[mask]

    if len(y_true) == 0:
        return {
            "n": 0,
            "bias": np.nan,
            "mae": np.nan,
            "rmse": np.nan,
            "r2": np.nan,
            "corr": np.nan,
        }

    bias = float((y_pred - y_true).mean())
    mae = float(mean_absolute_error(y_true, y_pred))
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    r2 = float(r2_score(y_true, y_pred)) if len(y_true) > 1 else np.nan
    corr = float(np.corrcoef(y_true, y_pred)[0, 1]) if len(y_true) > 1 else np.nan

    return {
        "n": int(len(y_true)),
        "bias": bias,
        "mae": mae,
        "rmse": rmse,
        "r2": r2,
        "corr": corr,
    }


def plot_timeseries(df, var, outdir):
    plot_df = df[["timestamp", f"{var}_ecowitt", f"{var}_reference"]].dropna(how="all")
    if plot_df.empty:
        print(f"No data to plot for {var}")
        return

    plt.figure(figsize=(14, 5))
    plt.plot(plot_df["timestamp"], plot_df[f"{var}_ecowitt"], label="Ecowitt")
    plt.plot(plot_df["timestamp"], plot_df[f"{var}_reference"], label="Reference")
    plt.title(f"Time series comparison: {var}")
    plt.xlabel("Time")
    plt.ylabel(var)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(outdir, f"timeseries_{var}.png"), dpi=150)
    plt.show()


def plot_scatter(df, var, outdir):
    plot_df = df[[f"{var}_reference", f"{var}_ecowitt"]].dropna()
    if plot_df.empty:
        print(f"No data to plot for {var}")
        return

    x = plot_df[f"{var}_reference"]
    y = plot_df[f"{var}_ecowitt"]

    plt.figure(figsize=(6, 6))
    plt.scatter(x, y, alpha=0.6)
    min_v = np.nanmin([x.min(), y.min()])
    max_v = np.nanmax([x.max(), y.max()])
    plt.plot([min_v, max_v], [min_v, max_v], linestyle="--")
    plt.title(f"Scatter comparison: {var}")
    plt.xlabel("Reference")
    plt.ylabel("Ecowitt")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(outdir, f"scatter_{var}.png"), dpi=150)
    plt.show()



def plot_single_series(df, time_col, value_col, title, ylabel, outpath):
    tmp = df[[time_col, value_col]].dropna()
    if tmp.empty:
        print(f"Skipping {value_col}: no data")
        return

    plt.figure(figsize=(12, 5))
    plt.plot(tmp[time_col], tmp[value_col])
    plt.title(title)
    plt.xlabel("Time")
    plt.ylabel(ylabel)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(outpath, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()


def plot_comparison_timeseries(df, time_col, eco_col, ref_col, logical_var, outpath):
    tmp = df[[time_col, eco_col, ref_col]].dropna()
    if tmp.empty:
        print(f"Skipping comparison timeseries for {logical_var}: no overlapping data")
        return

    plt.figure(figsize=(12, 5))
    plt.plot(tmp[time_col], tmp[eco_col], label="Ecowitt")
    plt.plot(tmp[time_col], tmp[ref_col], label="Reference")
    plt.title(f"Comparison time series: {logical_var}")
    plt.xlabel("Time")
    plt.ylabel(logical_var)
    plt.legend()
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(outpath, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()


def plot_scatter_comparison(df, eco_col, ref_col, logical_var, outpath):
    tmp = df[[eco_col, ref_col]].dropna()
    if tmp.empty:
        print(f"Skipping scatter for {logical_var}: no overlapping data")
        return

    plt.figure(figsize=(6, 6))
    plt.scatter(tmp[ref_col], tmp[eco_col], alpha=0.6)

    mn = min(tmp[ref_col].min(), tmp[eco_col].min())
    mx = max(tmp[ref_col].max(), tmp[eco_col].max())
    plt.plot([mn, mx], [mn, mx])

    plt.xlabel("Reference")
    plt.ylabel("Ecowitt")
    plt.title(f"Scatter comparison: {logical_var}")
    plt.tight_layout()
    plt.savefig(outpath, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()

# =========================
# Aggregation rules for resampling
# =========================
AGG_RULES = {
    "outdoor_temp": "mean",
    "outdoor_humidity": "mean",
    "dew_point": "mean",
    "feels_like": "mean",
    "wind_speed": "mean",
    "wind_gust": "max",
    "pressure": "mean",
    "rain_rate": "mean",
    "daily_rain": "max",
    "solar_radiation": "mean",
    "uv_index": "max",
}

In [22]:
#@title 5) Load data from repo

ecowitt_path = os.path.join(REPO_DIR, ECOWITT_FILE)
#reference_path = os.path.join(REPO_DIR, REFERENCE_FILE)

ecowitt_raw = robust_read_csv(ecowitt_path)
#reference_raw = robust_read_csv(reference_path)

print("Ecowitt shape:", ecowitt_raw.shape)
#print("Reference shape:", reference_raw.shape)

display(ecowitt_raw.head())
#display(reference_raw.head())

Ecowitt shape: (2234, 15)


,timestamp,outdoor_temp,outdoor_humidity,dew_point,feels_like,wind_speed,wind_gust,wind_dir,pressure,rain_rate,daily_rain,solar_radiation,uv_index,indoor_temp,indoor_humidity
0,2025-12-22T18:01:17.609Z,26.0,41.0,11.7,26.0,0.0,0.0,202.0,29.92,0.0,0.0,2.4,0.0,24.9,42.0
1,2025-12-22T19:01:17.852Z,26.3,40.0,11.6,26.3,0.0,0.0,127.0,29.91,0.0,0.8,2.1,0.0,26.1,40.0
2,2025-12-22T20:01:17.232Z,26.9,36.0,10.6,26.6,0.0,0.0,245.0,29.90,0.0,0.3,1.9,0.0,26.7,37.0
3,2025-12-22T21:01:18.421Z,26.8,36.0,10.5,26.5,0.0,0.0,245.0,29.91,0.0,0.3,1.7,0.0,26.4,36.0
4,2025-12-22T22:01:14.613Z,26.5,35.0,9.8,26.5,0.0,0.0,245.0,29.91,0.0,0.3,1.4,0.0,26.1,35.0


In [23]:
#@title 6) Inspect columns

print("Ecowitt columns:")
print(list(ecowitt_raw.columns))
#print("\\nReference columns:")
#print(list(reference_raw.columns))

Ecowitt columns:
['timestamp', 'outdoor_temp', 'outdoor_humidity', 'dew_point', 'feels_like', 'wind_speed', 'wind_gust', 'wind_dir', 'pressure', 'rain_rate', 'daily_rain', 'solar_radiation', 'uv_index', 'indoor_temp', 'indoor_humidity']


In [27]:
#@title 7) Clean and standardize datasets

ecowitt = standardize_time(ecowitt_raw, ECOWITT_TIME_COL, timezone=TIMEZONE)
#reference = standardize_time(reference_raw, REFERENCE_TIME_COL, timezone=TIMEZONE)

ecowitt = convert_numeric_columns(ecowitt, exclude=[ECOWITT_TIME_COL])
#reference = convert_numeric_columns(reference, exclude=[REFERENCE_TIME_COL])

ecowitt = apply_date_filter(ecowitt, ECOWITT_TIME_COL, '2025', '2026')
#reference = apply_date_filter(reference, REFERENCE_TIME_COL, START_DATE, END_DATE)



#for logical_var, func in UNIT_CONVERSIONS.get("reference", {}).items():
 #   col = VARIABLE_MAP[logical_var]["reference"]
  #  if col in reference.columns:
   #     reference[col] = func(reference[col])

print("Cleaned Ecowitt:", ecowitt.shape)
#print("Cleaned Reference:", reference.shape)
display(ecowitt.head())
#display(reference.head())

Cleaned Ecowitt: (222, 15)


,timestamp,outdoor_temp,outdoor_humidity,dew_point,feels_like,wind_speed,wind_gust,wind_dir,pressure,rain_rate,daily_rain,solar_radiation,uv_index,indoor_temp,indoor_humidity
0,2025-12-22 18:01:17.609000+00:00,26.0,41.0,11.7,26.0,0.0,0.0,202.0,29.92,0.0,0.0,2.4,0.0,24.9,42.0
1,2025-12-22 19:01:17.852000+00:00,26.3,40.0,11.6,26.3,0.0,0.0,127.0,29.91,0.0,0.8,2.1,0.0,26.1,40.0
2,2025-12-22 20:01:17.232000+00:00,26.9,36.0,10.6,26.6,0.0,0.0,245.0,29.90,0.0,0.3,1.9,0.0,26.7,37.0
3,2025-12-22 21:01:18.421000+00:00,26.8,36.0,10.5,26.5,0.0,0.0,245.0,29.91,0.0,0.3,1.7,0.0,26.4,36.0
4,2025-12-22 22:01:14.613000+00:00,26.5,35.0,9.8,26.5,0.0,0.0,245.0,29.91,0.0,0.3,1.4,0.0,26.1,35.0


In [40]:
#@title 8) Resample to common timestep

ecowitt_col_map = {k: v["ecowitt"] for k, v in VARIABLE_MAP.items()}
reference_col_map = {k: v["reference"] for k, v in VARIABLE_MAP.items()}

ecowitt_resampled = resample_dataset(
    ecowitt,
    ECOWITT_TIME_COL,
    ecowitt_col_map,
    RESAMPLE_FREQ,
    AGG_RULES,
    dataset_name="Ecowitt",
)

#reference_resampled = resample_dataset(
 #   reference,
  #  REFERENCE_TIME_COL,
   # reference_col_map,
    #RESAMPLE_FREQ,
    #AGG_RULES,
    #dataset_name="Reference",
#)

ecowitt_resampled = ecowitt_resampled.rename(columns={c: f"{c}_ecowitt" for c in ecowitt_resampled.columns if c != ECOWITT_TIME_COL})
#reference_resampled = reference_resampled.rename(columns={c: f"{c}_reference" for c in reference_resampled.columns if c != REFERENCE_TIME_COL})

ecowitt_resampled = ecowitt_resampled.rename(columns={ECOWITT_TIME_COL: "timestamp"})
#reference_resampled = reference_resampled.rename(columns={REFERENCE_TIME_COL: "timestamp"})

#comparison = pd.merge(ecowitt_resampled, reference_resampled, on="timestamp", how="inner")

#print("Merged comparison shape:", comparison.shape)
#display(comparison.head())

In [41]:
#@title 9) Compute metrics

metrics_rows = []

for logical_var in VARIABLE_MAP.keys():
    eco_col = f"{logical_var}_ecowitt"
    ref_col = f"{logical_var}_reference"

    if eco_col not in comparison.columns or ref_col not in comparison.columns:
        print(f"Skipping {logical_var}: missing merged columns")
        continue

    stats = compute_metrics(comparison[ref_col], comparison[eco_col])
    stats["variable"] = logical_var
    metrics_rows.append(stats)

metrics_df = pd.DataFrame(metrics_rows)[["variable", "n", "bias", "mae", "rmse", "r2", "corr"]]
display(metrics_df)

metrics_path = os.path.join(OUTPUT_DIR, "comparison_metrics.csv")
metrics_df.to_csv(metrics_path, index=False)
print("Saved:", metrics_path)

NameError: name 'comparison' is not defined

In [52]:
#@title 10) Generate plots

# Make sure these names match your actual dataframes
# ecowitt_resampled
# reference_resampled
# comparison

for logical_var in VARIABLE_MAP.keys():
    eco_var = logical_var
    ref_var = logical_var

    # 1) Ecowitt individual plot from ecowitt_resampled
    if eco_var in ecowitt_resampled.columns:
        plot_single_series(
            ecowitt_resampled,
            time_col=ECOWITT_TIME_COL,
            value_col=eco_var,
            title=f"Ecowitt: {logical_var}",
            ylabel=logical_var,
            outpath=os.path.join(OUTPUT_DIR, f"{logical_var}_ecowitt_timeseries.png"),
        )
    else:
        print(f"Skipping Ecowitt {logical_var}: column not found")

    # 2) Reference individual plot from reference_resampled
    #if ref_var in reference_resampled.columns:
     #   plot_single_series(
      #      reference_resampled,
       #     time_col=REFERENCE_TIME_COL,
        #    value_col=ref_var,
         #   title=f"Reference: {logical_var}",
          #  ylabel=logical_var,
           # outpath=os.path.join(OUTPUT_DIR, f"{logical_var}_reference_timeseries.png"),
        #)
    #else:
     #   print(f"Skipping Reference {logical_var}: column not found")

    # 3) Comparison plots from merged/comparison dataframe
    #eco_col_cmp = f"{logical_var}_ecowitt"
    #ref_col_cmp = f"{logical_var}_reference"

    #if eco_col_cmp in comparison.columns and ref_col_cmp in comparison.columns:
     #   plot_comparison_timeseries(
      #      comparison,
       #     time_col="time",   # change if your merged time column has another name
        #    eco_col=eco_col_cmp,
         #   ref_col=ref_col_cmp,
          #  logical_var=logical_var,
           # outpath=os.path.join(OUTPUT_DIR, f"{logical_var}_comparison_timeseries.png"),
        #)

        #plot_scatter_comparison(
         #   comparison,
          #  eco_col=eco_col_cmp,
           # ref_col=ref_col_cmp,
            #logical_var=logical_var,
            #outpath=os.path.join(OUTPUT_DIR, f"{logical_var}_scatter.png"),
        #)
    #else:
     #   print(f"Skipping comparison {logical_var}: merged columns not found")

Skipping Ecowitt temp_c: column not found
Skipping Ecowitt humidity_pct: column not found
Skipping Ecowitt dew_point_c: column not found
Skipping Ecowitt feels_like_c: column not found
Skipping Ecowitt wind_speed: column not found
Skipping Ecowitt wind_gust: column not found
Skipping Ecowitt pressure: column not found
Skipping Ecowitt rain_rate: column not found
Skipping Ecowitt daily_rain: column not found
Skipping Ecowitt solar_radiation: column not found
Skipping Ecowitt uv_index: column not found


In [43]:
#@title 11) Save merged comparison table

comparison_path = os.path.join(OUTPUT_DIR, "ecowitt_vs_reference_merged.csv")
comparison.to_csv(comparison_path, index=False)

print("Saved merged data to:", comparison_path)
print("Files in output directory:")
for p in sorted(Path(OUTPUT_DIR).glob("*")):
    print(" -", p.name)

NameError: name 'OUTPUT_DIR' is not defined